In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn import preprocessing
from sklearn.linear_model import RidgeCV
from sklearn import svm
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.compose import TransformedTargetRegressor
from sklearn.inspection import permutation_importance

# Machine learning inference of a demographic model based on summary statistics 

## A walkthrough classical steps

**Arnaud Quelin and Flora Jay flora.jay@lisn.fr**

We will see how classical tools of machine learning can help us infer evolutionary parameters

# 1. Load the reference table

<div class='alert-info'>
    
For the sake of speed we have done simulations beforehand using a coalescent simulator (msprime). They correspond to an isolation migration model (ancestral population splitting in two at $T_{split}$ followed by migration at a given rate between these two populations)

Simulation procedure:
- We drawn parameters for 2000 scenarios of population split followed by migration.
- For each scenario we wimulated 100 replicates using msprime
- The diploid sample size of population i is $n_i$ (so there are two times as many haplotypes). It's population size is $N_i$. The ancestral size is 1000.

</div>

In [ ]:
path = './'

In [ ]:
df = pd.read_csv(path + 'sumstats_all.zip')

In [ ]:
df.shape

In [ ]:
df.columns

# 2. Exploration

## Let's have a look at the simulation parameters

In [ ]:
df.columns[:11]

In [ ]:
df.iloc[:,:11].apply([min,max])

<div class="alert alert-warning">
<strong>Which parameters are fixed, which ones are varying?</strong>

Which ones are used to structure the dataset (e.g. index the simulations done, etc.)?
Which ones are used to actually control the coalescent simulation?

</div>

<div class="alert alert-info">
  Each time there's an orange box (and sometimes more), write your code (if needed) and a textual answer with your explanation below.
</div>

*YOUR ANSWER*

<div class="alert alert-warning">
<strong> Using sns.histplot display the distributions of the varying parameters</strong>

Why is the distribution called an empirical prior?

Given these plots, what do you think the priors were for each parameter?

</div>

In [ ]:
sns.histplot(df['T_split'], kde=True)

In [ ]:
# CODE BELOW FOR OTHER VARYING EVOLUTIONARY PARAMETERS

In [ ]:
sns.histplot(df['migration_rate'], kde=True)

## Let's have a look at the potential predictor variables

In [ ]:
df.columns[11:]

List of sumstats: <br>
- Between pop A and pop B:

    - Dxy: Mean pairwise genetic divergence for pairs (the average across every possible pair of chromosomes (one from each population))
    - Fst: Measure of differentiation between the 2 populations   
.
- Computed for pop A (indexed by '_0'), pop B (indexed by '_1') and pop A + pop B (indexed by '_2'):

    - S: nb of segregating sites
    - PI_mean, PI_std: diversity mean and standard deviation
    - D: Tajima D
    - winHET_mean, winHET_std: compute haplotypic heterozygosity in 50kbp windows sliding along the genome and return mean and varianceh
    
.
- SFS computed  for pop A, pop B and pop A + pop B (39 sumstats):     

    - SFS for population A: sfs_0 to sfs_9   ($2*n_1$values where $n_1$ = diploid sample size of pop A, since mutations can be fixed in popA but not in the overall sample (popA+popB)

    - SFS for population B: sfs_10 to sfs_19 ($2*n_2$  values)   

    - SFS for all samples (popA+popB): sfs_20 to sfs_38 (n-1 where n = total haploid sample size= $2*n_1 + 2*n_2$)   
    
    
[most stats have been computed using tskit](https://tskit.dev/tskit/docs/stable/stats.html)

<div class="alert alert-warning">

**Are we working with 'raw' data or handcrafted features?**
</div>

ANSWER

<div class="alert alert-warning">

What are the distribution of i) **Fst** and ii) the **diversity in pop A, B, and global population**? Explain the different distributions.
    
    
To display multiple distributions on one plot, you can use `sns.histplot(df[['name1', 'name2',..]], kde=True)`,
</div>

In [ ]:
sns.histplot(df[['PI_mean_0','PI_mean_1','PI_mean_2']], kde=True)

In [ ]:
# INSERT CODE HERE

In [ ]:
sns.histplot(df['Fst_0'], kde=True)

In [ ]:
ss_names = np.array(df.columns[11:],dtype='str')

In [ ]:
sns.histplot(df[ss_names[np.char.startswith(ss_names,'S_')]], kde=True)

In [ ]:
sns.histplot(df[ss_names[np.char.startswith(ss_names,'winHET_mean')]], kde=True)

<div class="alert alert-warning">
   Given the evolutionay models simulated, is the distribution of Fst expected? and these different distributions of diversity? Why?
</div>

ANSWER:

## Explore visually the relationship between the evolutionary parameters and some summary statistics 


<div class="alert alert-warning">
Explore visually the relationship between the demographic parameters and some summary statistics 
such as Tajima's D and  Fst
</div>

For ease of visualization we randomly subsample the reference table

In [ ]:
rnd_subset_index = np.random.choice(df.index,1000)
df_sub = df.loc[rnd_subset_index]
df.shape, df_sub.shape 

<div class="alert alert-info">
Example for Fst versus migration rate:
</div>

In [ ]:
plt.plot(df_sub['migration_rate'], df_sub['Fst_0'], '.')
plt.xlabel('migration_rate')
plt.ylabel('Fst_0')

In [ ]:
np.corrcoef(df_sub['migration_rate'], df_sub['Fst_0'])

<div class="alert alert-warning">
Did you expect to observe this trend and this correlation value?
    
Why is the correlation value negative, and also why aren't the two variables perfectly correlated?
</div>

In [ ]:
# Let's add a fitted linear regression line on the plot, 
# There's a function that combine both the regression and the plot for you!
sns.regplot(data=df_sub, x='migration_rate', y='Fst_0', line_kws={"color": "C1"})

plt.title("Linear regression")

<div class="alert alert-warning">
INSERT CODE BELOW TO INVESTIGATE OTHER SUMMARY STATS / OTHER TARGET PARAMETERS RELATIONSHIPS

</div>

In [ ]:
# you can repeat what was done above or 
# sns.PairGrid can help you doing things less manually, example:
# g = sns.PairGrid(df_sub, y_vars=['Fst_0','sfs_mean_0'], x_vars=['migration_rate','T_split'])    
# g = g.map(sns.regplot,line_kws={"color": "C1"})

In [ ]:
g = sns.PairGrid(df_sub, y_vars=['Fst_0','D_2','sfs_mean_0'], x_vars=['migration_rate','T_split'])    
g = g.map(sns.regplot,line_kws={"color": "C1"},order=1)

In [ ]:
# For all summary statistics but SFS
g = sns.PairGrid(df_sub, 
                 y_vars=ss_names[~np.char.startswith(ss_names,'sfs')], 
                 x_vars=['migration_rate','T_split'])    
g = g.map(sns.regplot,line_kws={"color": "C1"},order=1)

# 3. Setting up variables; splitting dataset into train, validation, test

In [ ]:
par_names = np.array(df.columns[2:11],dtype='str')
ss_names = np.array(df.columns[11:],dtype='str') # We did that already, for practicality I redo it here in case I have skipped the first part of the tuto

In [ ]:
u_sc = np.unique(df['scenario_idx'])
train_sc, non_train_sc = train_test_split(u_sc,train_size=0.7, test_size=0.3, random_state=42) # randomly partition in 2
val_sc, test_sc = train_test_split(non_train_sc,train_size=0.5, test_size=0.5, random_state=42) # partition one of the 2 previous set in 2

print(f"*Full set many replicates per scenario:\n")

print('train size:', train_sc.shape[0], 'scenarios')
print('val size:', val_sc.shape[0], 'scenarios')
print('test size:', test_sc.shape[0], 'scenarios')


df_train = df[df['scenario_idx'].isin(train_sc)]
df_val = df[df['scenario_idx'].isin(val_sc)]
df_test = df[df['scenario_idx'].isin(test_sc)]
print('train size:', df_train.shape[0], 'total examples')
print('val size:', df_val.shape[0], 'total examples')
print('test size:', df_test.shape[0], 'total examples')


# Create a subset split with less replicate per scenario to do tests faster
nrep = 5
print(f"\n\n*Subsets with {nrep} replicates per scenario:\n")
mask = df['replicate_idx']<nrep
dfsub_train = df[ (df['scenario_idx'].isin(train_sc)) & mask ]
dfsub_val = df[ (df['scenario_idx'].isin(val_sc))  & mask ]
dfsub_test = df[ (df['scenario_idx'].isin(test_sc))  & mask ]
print('train size:', dfsub_train.shape[0], 'total examples')
print('val size:', dfsub_val.shape[0], 'total examples')
print('test size:', dfsub_test.shape[0], 'total examples')



<div class="alert alert-warning">
<strong>
    
- Why splitting the dataset in 3? What is the use of each set?

- What's the difference between dfsub_* datasets and df_* datatsets? Why did we build dfsub_* ?
</strong>

</div>

ANSWER:

<div class="alert alert-warning">
Our goal will be to the predict demographic parameters from the summary statistics.

- Is it a supervised or unsupervised task, and why?

- Would you design the task as a regression or a classification, and why?

</div>

ANSWER:

# 4. Predicting the migration rate (or another demographic parameter) with linear regression


Generic scikit-learn syntax:

    
    # define the model you'll want to use
    # model = EstimatorName(hyperparameters)
    model = LinearRegression()  # for example
    
    # Fit the model to your data, i.e. learn its optimizable weights/parameters
    # if X is the observable data
    # y the  associated label you will want to predict (continuous for regression; categorical for classification)
    model.fit(X,y)
    
    # Get the score associated to the statistical model used
    model.score(X,y) # on train data
    model.score(Xval, yval) # on validation data
    
    # Predict the labels for another dataset (for you don't know the labels)
    model.predict(Xnew)
    
    # Plot results, compute other scores, etc
    ...
    
[more estimators](https://scikit-learn.org/stable/glossary.html#term-estimators)

[scikit-learn getting started](https://scikit-learn.org/stable/getting_started.html)

In [ ]:
ss = ['D_2']  # choosing a subset of the summary statistics
target = 'migration_rate'  # choosing a target parameter (the one that has to be predicted)
X = df_train[ss]
y = df_train[target]
reg = LinearRegression().fit(X, y)
print('coef of determination train:', reg.score(X, y))
print('coef of determination validation:', reg.score(df_val[ss], df_val[target]))
print(reg.score(X, y), reg.coef_, reg.intercept_)
pred = reg.predict(df_val[ss])
print('rmse:', np.mean((pred - df_val[target])**2))

<div class="alert alert-warning">
    
What is the score outputted by the LinearRegression() model? (look at the documentation)
    
How is it linked to RMSE ?
     
A good model should have a low or high coefficient of determination? a low or high RMSE?
</div>

See  [score details](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression.score)


ANSWER:

<div class="alert alert-warning">
Based on the above, is D_2 a good predictor of the migration rate?
</div>

<div class="alert alert-warning">
You could also plot the predicted values as a function of the true ones (see cell below) What do you see?
</div>

In [ ]:
#plot last model prediction for validation set
score = reg.score(df_val[ss], df_val[target])
plt.plot(df_val[target], reg.predict(df_val[ss]), '.', alpha=.1)
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on validation set: {score} \n using {ss} ')

<div class="alert alert-warning">
<strong> Perform a linear regression using (i) Fst ('Fst_0') and (ii) using all summary statistics jointly, and write your interpretation of the results </strong>

</div>

In [ ]:
# INSERT CODE BELOW

In [ ]:
ss = ['Fst_0']
target = 'migration_rate'
X = df_train[ss]
y = df_train[target]
reg = LinearRegression().fit(X, y)
print('coef of determination train:', reg.score(X, y))
print('coef of determination validation:', reg.score(df_val[ss], df_val[target]))
# print(reg.score(X, y), reg.coef_, reg.intercept_)
pred = reg.predict(df_val[ss])
print('rmse:', np.mean((pred - df_val[target])**2))

In [ ]:
# Linear regression using all summary statistics
ss = ss_names
target = 'migration_rate'
X = df_train[ss]
y = df_train[target]
reg = LinearRegression().fit(X, y)
print('coef of determination train:', reg.score(X, y))
print('coef of determination validation:', reg.score(df_val[ss], df_val[target]))
# print(reg.score(X, y), reg.coef_, reg.intercept_)
pred = reg.predict(df_val[ss])
print('rmse:', np.mean((pred - df_val[target])**2))

<div class="alert alert-info">
<strong> For the validation set, here is how to plot the predicted values of a trained model stored in 'reg' as a function of the expected ones.</strong>

</div>

In [ ]:
#plot last model prediction for validation set
score = reg.score(df_val[ss], df_val[target])
print("sumstats used:", ss)
plt.plot(df_val[target], pred, '.', alpha=.2)
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on validation set: {score}')

<div class="alert alert-warning">
Your interpretation:
    </div>

### Time of split

<div class="alert alert-warning">
<strong> Repeat the analysis to predict the divergence time using a linear regression of all summary statistics</strong>

Compare performances with the migration rate.
</div>

In [ ]:
# INSERT CODE BELOOW

In [ ]:
# Linear regression using all summary statistics
ss = ss_names
target = 'T_split'
X = df_train[ss]
y = df_train[target]
reg = LinearRegression().fit(X, y)
print('coef of determination train:', reg.score(X, y))

score = reg.score(df_val[ss], df_val[target])
print('coef of determination validation:', score)
# print(reg.score(X, y), reg.coef_, reg.intercept_)
pred = reg.predict(df_val[ss])
print('rmse:', np.mean((pred - df_val[target])**2))

#plot last model prediction for validation set
print("sumstats used:", ss)
plt.plot(df_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on validation set: {score}')

# 5. Generic run_model function

In [ ]:
# Let us write a small generic function - be sure to read it
def run_model(mod, ss, target, df_train=df_train, df_val=df_val):
    # pipe = make_pipeline(StandardScaler(), mod)
    pipe = mod
    pipe.fit(df_train[ss], df_train[target])
    pred = pipe.predict(df_val[ss])
    score = pipe.score(df_val[ss], df_val[target])
    print(f'score on validation set {score}')
    rmse = np.mean((pred - df_val[target])**2)
    # print(f'rmse {rmse}')
    return (score, rmse, pred, pipe)


In [ ]:
# example:
score, rmse, pred, model = run_model(LinearRegression(), ss_names, 'migration_rate')

In [ ]:
model

# 6. -skip-  Linear ridge regression 
(you can go back to this at the end of the session if you have more time)

<div class="alert alert-warning">
*skip today - go back at the end of the session if you have time*
    
What is the main hyperparameter of a ridge regression? What's is use?
    
You can vary it and find the one leading to the smallest error (this is called hyperparameter optimization) and should never be done using the final test set. Indeed the test set allows to check at the very end that we have not overfit the training set (used to optimize the model weights) or validation set (usually used for hyperparameter tuning).
    
</div>

In [ ]:
run_model(Ridge(alpha=1e-3), ss_names, 'migration_rate')

In [ ]:
alphas=[1e-14,1e-12,1e-8,1e-6,1e-4,1e-2,1e-1,1]
rmses=dict()
scores=dict()
for alpha in alphas:
    reg= make_pipeline(Ridge(alpha=alpha))
    score, rmse, _, _ = run_model(reg, ss_names, 'migration_rate')
    rmses[alpha]=rmse
    scores[alpha]=score

plt.plot(alphas, scores.values(),'o')
plt.xlabel('alpha')
plt.ylabel('score')

plt.xscale('log')


In [ ]:
# There is actually a wrapper to do something similar to the loop for you
mod = RidgeCV(alphas=alphas).fit(X, y)
print("alpha, factor for regularizing term, chosen by cross-validation: ",mod.alpha_)

# 7. Regression with Random Forest - and hyperparameter optimization
Random forest is an ensemble method that averages prediction accross weak predictors (simple tree classifier).

It can be used both for classification and for regression.

For detailed description of the options and hyperparameters, refer to [the scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html)

**We will train the models on a subset of the data for speed. Feel free to change to the whole set later on, when your setups are correct**

## Intro: training on a subset and visualization of one of the trained tree

<div class="alert alert-info">

Let us first train a RF on small subset of the summary statistics and a very small subset of the train, so that we can dissect one of the trained tree of the ensemble visually (displaying the tree does not scale well with large datasets!):
    
</div>

In [ ]:
from sklearn.tree import plot_tree

In [ ]:
# For each tree and each sample there will be max_depth decision done at max
# Overall there will be n_estimators trees  trained, using n_jobs
rf = RandomForestRegressor(n_estimators = 10, n_jobs = 4, max_depth=3) 

target = 'migration_rate'
ss = ['PI_mean_0','PI_mean_1','PI_mean_2','Fst_0','S_0','S_1','S_2','D_0','D_1','D_2']  
# Like for SVM, RF are able to handle an important number of input features 
# So in real experiments you would keep all summary statistics but we select her for the sake of the tuto (easier visualization)

score, rmse, pred, model = run_model(rf, 
                                     ss, 
                                     target,df_train.iloc[np.random.randint(df_train.shape[0], size=100)],  # use only 100 samples to build tree
                                     dfsub_val)

fig = plt.figure(figsize=(4,4))
plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score: {score}')

In [ ]:
# Here you can see that the rf is composed of
print(f"This random forest is composed of {len(rf.estimators_)} \n") 
print("Here are the first five:\n", rf.estimators_[:5])

In [ ]:
# Let's visualize the structure of the first tree
# You can control the number of decisions you want to display (ie how far down in the tree you want to go) with the max_depth argument below
tree = rf.estimators_[0]
fig = plt.figure(figsize=(15,7))
_ = plot_tree(tree, feature_names=rf.feature_names_in_, filled=True, impurity=False, precision=5, max_depth=None)

In [ ]:
xval, yval = df_val[ss].iloc[0:1,:], df_val[target].iloc[0]
print('The sample picked has for summary statistics:\n', xval.T)
print(f"\nThe first tree of the RF predicts {target}= {tree.predict(np.array(xval))} for this sample") 
print(f"The whole RF predicts {target}= {rf.predict(xval)} for this sample") 
print(f"The true label is {target}= {yval} for this sample") 

<div class="alert alert-warning">
Take some time to understand the analyses above. You can have fun double-checking that the first tree (displayed above) is correct for the sample picked above. Note that the computer should be faster than you at this job ;) 
</div>

## Training a random forest for regression

### The role of hyperprameters

<div class="alert alert-info">

Let us now train a RF on all summary statistics, which should hopefully yield better results.

In theory, we should train on the full training set to improve performance (but for speed and the sake of the tuto we can keep dfsub for now)
    
</div>

In [ ]:
rf = RandomForestRegressor(n_estimators = 10, n_jobs = 4, max_depth=100)

target = 'migration_rate'
score, rmse, pred, model = run_model(rf, ss_names, target ,dfsub_train, dfsub_val)

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score: {score}')

<div class="alert alert-warning">

What is the main hyperparameters (HP) of a random forest?  (See documentation, or do maj-tab on the function name in the cell above)
    
They can be varied in order to find the HP values leading to the smallest error (this is called hyperparameter optimization) and should never be done using the final test set. Indeed the test set allows to check at the very end that we have not overfitted the training set (used to optimize the model weights) or the validation set (usually used for hyperparameter tuning).
    
(Note that the previous methods such as ridge regression and SVM also had hyperparameters).
    
</div>

ANSWER:

<div class="alert alert-warning">
    
Investigate whether two hyperparameters, **max_features** and **n_estimators**, have an impact on the performances and on running time 
    
You can use the cell magic `%%time` at the beginning of a cell to time the cell execution. Or `%time` in front of a specific line
    
</div>

<div class="alert alert-warning">
    
Let's do some manual checks by simply changing the values for these parameters in the previous code (later we will do it more automatically). E.g. start with increasing n_estimators from 10 to 100 trees, what do you observe?
</div>

In [ ]:
# INSERT CODE BELOW

In [ ]:
%%time
n_estimators = 10
rf = RandomForestRegressor(n_estimators = n_estimators, n_jobs = 4)

target = 'migration_rate'
score, rmse, pred, model = run_model(rf, ss_names, target ,dfsub_train, dfsub_val)

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'n_est {n_estimators}; score: {score}')

In [ ]:
%%time
n_estimators = 100
rf = RandomForestRegressor(n_estimators = n_estimators, n_jobs = 4)

target = 'migration_rate'
score, rmse, pred, model =  run_model(rf, ss_names, target ,dfsub_train, dfsub_val)

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'n_est {n_estimators}; score: {score}')

In [ ]:
%%time
n_estimators = 10
max_features='log2'

rf = RandomForestRegressor(n_estimators = n_estimators, max_features=max_features,n_jobs = 4)

target = 'migration_rate'
score, rmse, pred, model = run_model(rf, ss_names, target ,dfsub_train, dfsub_val)

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'n_est {n_estimators}; max_features {max_features}; score: {score}')

In [ ]:
%%time

n_estimators = 100
max_features='log2'

rf = RandomForestRegressor(n_estimators = n_estimators, max_features=max_features,n_jobs = 4)

target = 'migration_rate'
score, rmse, pred, model = run_model(rf, ss_names, target ,dfsub_train, dfsub_val)

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'n_est {n_estimators}; max_features {max_features}; score: {score}')


<div class="alert alert-info">
    
Now, let's do this more automatically, e.g. for n_estimators:

    
</div>

In [ ]:
max_features=1.0
scores=dict()
for n_estimators in [2, 5, 10, 20, 100, 250, 500]:
    print(f'{n_estimators} trees:')

    rf = RandomForestRegressor(n_estimators = n_estimators, max_features=max_features,n_jobs = 4)
    target = 'migration_rate'
    %time score, rmse, pred, model = run_model(rf, ss_names, target ,dfsub_train, dfsub_val)
    scores[n_estimators]=score
    

In [ ]:
plt.plot(scores.keys(),scores.values(),'o')
plt.xlabel('n_estimators')
plt.ylabel('score')
plt.title(f'n_estimators={max(scores, key=scores.get)} yields highest score among tested values')

In [ ]:
max_features='log2'
scores2=dict()
for n_estimators in [2, 5, 10, 20, 100, 250, 500]:
    print(f'{n_estimators} trees:')

    rf = RandomForestRegressor(n_estimators = n_estimators, max_features=max_features,n_jobs = 4)
    target = 'migration_rate'
    %time score, rmse, pred, model = run_model(rf, ss_names, target ,dfsub_train, dfsub_val)
    scores2[n_estimators]=score
    

In [ ]:
plt.plot(scores2.keys(),scores2.values(),'o')
plt.xlabel('n_estimators')
plt.ylabel('score')
plt.title(f'n_estimators={max(scores2, key=scores2.get)} yields highest score among tested values')

<div class="alert alert-warning">

According to these results, what hyperparameter should we use?

</div>

ANSWER:

### -skip-   RF regression HP grid search
Instead of the "manual" option above, there are functions in scikit-learn that allows you to do HP grid search a bit more automatically [see full doc](https://scikit-learn.org/stable/modules/grid_search.html)

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {'max_depth': [3, 5],
               'min_samples_split': [5, 10]}
base_estimator = RandomForestRegressor(random_state=0, n_estimators=10)

X, y = dfsub_train[ss_names], dfsub_train[target]
sh = GridSearchCV(base_estimator, param_grid, cv=2, n_jobs=4).fit(X, y)  
# cv was set to 2 for time sake, it's otherwise safer to use the default (5)

sh.best_estimator_


In [ ]:
RandomForestRegressor(max_depth=sh.best_estimator_.max_depth, 
                      min_samples_split=sh.best_estimator_.min_samples_split,
                      n_estimators=sh.best_estimator_.n_estimators, random_state=0)

<div class="alert alert-warning">
skip today
    
Perform a grid search on for hyperparameters you are interested in, then evaluate the performances on the validation set
</div>

In [ ]:
# INSERT CODE BELOW

### -skip-  RF regression for divergence time

In [ ]:
# INSERT CODE BELOW (maybe not today, for the sake of tutorial length)

In [ ]:
%%time
# Using the whole training set
n_estimators = 100
max_features='log2'

rf = RandomForestRegressor(n_estimators = n_estimators, max_features=max_features,n_jobs = 4)

target = 'T_split'
score, rmse, pred, model = run_model(rf, ss_names, target ,df_train, df_val)

plt.plot(df_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'n_est {n_estimators}; max_features {max_features}; score: {score}')


## RF feature importance

Interestingly RF gives you the opportunity to measure how much each input feature (summary statistic here) contributes to the prediction.


From the documentation:

The relative rank (i.e. depth) of a feature used as a decision node in a tree can be used to assess the relative importance of that feature with respect to the predictability of the target variable. **Features used at the top of the tree contribute to the final prediction decision of a larger fraction of the input samples.** The expected fraction of the samples they contribute to can thus be used as an estimate of the relative importance of the features. In scikit-learn, the **fraction of samples a feature contributes to is combined with the decrease in impurity from splitting them to create a normalized estimate of the predictive power of that feature.**

In [ ]:
# plot most important nfeat features
# rf = rf_saved
nfeat=25 # nb of top features to be plotted
sorted_idx = rf.feature_importances_.argsort()[::-1][:nfeat]
plt.barh(rf.feature_names_in_[sorted_idx][::-1], rf.feature_importances_[sorted_idx][::-1])
plt.xlabel("Random Forest Feature Importance")
title = str()
plt.title('Ranking of features importance for the prediction of '+ target)
plt.show()


<div class="alert alert-warning">
    
- Comments on the "winner" of this "feature importance competition"?
    
- What does this feature importance analysis bring compare to visualizing the tree as we had done previously?
</div>

<div class="alert-info">
    !BREAK! 
    
You've reached the end of the morning session. If you're done with the first part and have time you can either go back to the -skip- sections to learn about ridge regression (a penalized version of linear regression) and automatic HP tuning for RF. Or start the next part.
</div>

# 8. A change in task modeling: Classification with Support Vector Machine (SVM)

<div class="alert-info">
    
**We will now show that modeling the task is an important and flexible part of the work.** The researcher can intervene upstream and decide/design the task based on the exact question they need to ask / hypothesis they want to test, and based on their knowledge of the problem.

We will change the modeling and go **from a regression to a classification task**. For that, we will set up a **threshold for the target parameter**:

Under the threshold, the scenario is considered in the 'low' class and above it, it is in the 'high' class. For example, for migration rate the 2 classes will correspond to **low gene flow** and **high gene flow**.

We use this opportunity to apply another ML tool: **Support Vector Machines**, but we could also have used random forests (as above but for classification instead of regression)

**We will train the models on a subset of the data for speed. Feel free to change to the whole set later on when your setups are correct**
</div>

## Training and performances

### SVM on a subset of the summary statistics

In [ ]:
%time

# Selecting part of the input features (for illustrative purposes)
ss = ['S_0','PI_mean_2'] #ss_names
X = dfsub_train[ss]

# Computing the class labels (categorical variable) from the continuous parameter:
target = 'migration_rate'
threshold = df_train[target].median()
y = (dfsub_train[target] > threshold ) # transform the continuous value into a boolean (below or above the threshold?)
print(f"threshold for {target} was set to the empirical prior median: {threshold} \n lower values are labeled False, the other are labeled True")

# Fit an SVM
clf = svm.SVC()
clf.fit(X, y)

# Predict labels for the validation set
pred = clf.predict(dfsub_val[ss])

# What where the expected labels for the validation set?
# (also need to convert from continuous to categorical
target_val = dfsub_val[target] # continuous
val_labels = pd.Series( target_val > threshold, name='expected', index=target_val.index) # binary
pred = pd.Series(pred,name='predicted', index=target_val.index) # index the predicted table with expected labels

# Compute confusion matrix
cr = pd.crosstab(val_labels, pred)
cr

<div class="alert alert-warning">
    
- What is show in the confusion matrix cr? On what data was it computed?

- What numbers correspond to correct predictions?

- What is your interpretation, do we have a near perfect model?    


    
</div>

ANSWER

In [ ]:
# You can actually compute the misclassification rate as follows:
print( 'Misclassification rate is:', 1 - np.diag(cr).sum()/ cr.values.sum())

<div class="alert alert-warning">
    
The score that is most often use though, to account for class imbalance etc. is the F1 score
    
- $F_1 =  \frac{2 TP}{(2 TP + FP + FN)} $
    
    TP: True Positive;     
    FP: False Positive;  
    FN: False Negative;

- Identify TP,FP and FN in the confusion matrix displayed above and compute yourself (i.e. without importing a function) the f1 score from the confusion matrix above by identifying where are TP, FP, and FN values.

- Then double check using the function from sklearn.metrics that you got the correct score

- Why using this score and not the mse or determination coefficient?

    
</div>

In [ ]:
# fill in the ? values by reading the confusion matrix above, then uncomment the lines
"""
TP=?
FP=?
FN=?
2*TP/(2*TP + FP + FN)
"""

In [ ]:
# compare to the value obtain with the f1_score function
from sklearn.metrics import f1_score  # load function

ANSWER:

In [ ]:
f1_score(val_labels, pred)


<div class="alert alert-warning">

If you have time, train a SVM using all sumstats. Does the performance increase?
    
</div>

## -skip-  Training other SVMs

<div class="alert alert-warning">

Extra: train a SVM using all summary statistics; using a different kernel; using the linear SVM function directly)
    
</div>

#### -skip-  SVM on all summary statistics

In [ ]:
# INSERT CODE BELOW 

In [ ]:
%time

### USE ALL SUMSTATS ###

ss = ss_names
target = 'migration_rate'
threshold = df_train[target].median()
print(f"threshold for {target} is set to the prior median: {threshold}")

X = dfsub_train[ss]
y = (dfsub_train[target] > threshold )
clf = make_pipeline(StandardScaler(), svm.SVC())
clf.fit(X, y)


target_val = dfsub_val[target]
val_labels = pd.Series( target_val > threshold, name='expected', index=target_val.index)
pred = clf.predict(dfsub_val[ss])
pred = pd.Series(pred,name='predicted', index=target_val.index)
cr = pd.crosstab(val_labels, pred)
cr



In [ ]:
f1_score(val_labels, pred)


#### -skip-  Linear SVM on all summary statistics

In [ ]:
# INSERT CODE BELOW 

In [ ]:
%time
ss = ss_names
target = 'migration_rate'
threshold = df_train[target].median()
print(f"threshold for {target} is set to the prior median: {threshold}")

X = dfsub_train[ss]
y = (dfsub_train[target] > threshold )
clf = make_pipeline(StandardScaler(), svm.SVC(kernel='linear')) #svm.LinearSVC())
clf.fit(X, y)


target_val = dfsub_val[target]
val_labels = pd.Series( target_val > threshold, name='expected', index=target_val.index)
pred = clf.predict(dfsub_val[ss])
pred = pd.Series(pred,name='predicted', index=target_val.index)
cr = pd.crosstab(val_labels, pred)
cr



In [ ]:
f1_score(val_labels, pred)


#### -skip-  Linear SVM on all summary statistics and full dataset (~ 1minute to run) 

In [ ]:
# INSERT CODE BELOW 

In [ ]:
%%time

ss = ss_names
target = 'migration_rate'
threshold = df_train[target].median()
print(f"threshold for {target} is set to the prior median: {threshold}")

X = df_train[ss]
y = (df_train[target] > threshold )

clf = make_pipeline(StandardScaler(), svm.LinearSVC())
clf.fit(X, y)


target_val = df_val[target]
val_labels = pd.Series( target_val > threshold, name='expected', index=target_val.index)
pred = clf.predict(df_val[ss])
pred = pd.Series(pred,name='predicted', index=target_val.index)
cr = pd.crosstab(val_labels, pred)
cr



In [ ]:
f1_score(val_labels, pred)


## Performance and modelling question

<div class="alert alert-warning">
    
- We plot below the probability of being predicted in the high migration rate class (True) as a function of 20 bins of the true migration rate.
    
- Explain the plot

- Are the results satisfying? Do you think the problem is well-posed (i.e. the task correctly defined) ?    
    
</div>

In [ ]:
from scipy.stats import binned_statistic
bin_means, bin_edges, binnumber = binned_statistic(target_val, pred,  bins=20)
plt.figure()
plt.hlines(bin_means, bin_edges[:-1], bin_edges[1:], colors='g', lw=5,
            label='binned statistic of data')
plt.xlabel(target)
plt.ylabel(f"Probability of being predicted \n as having high gene flow")
plt.title(f'Prediction binned per {target} value')
plt.ylim(0,1)

ANSWER:

# 9. Impact of model misspecifications

<div class="alert-info">
    
We will load two smaller tables corresponding to **novel experiments**. They contain the simulation parameters and the output summary statistics.

But as you can guess from the title, there is a trick: the evolutionary models of the novel experiments do not match exactly the ones from the training set.

**--> The idea is now to check how well a statistical model trained on our previous dataset can predict labels for these novel experiments.**

We are back to using regression models.

</div>

## Loading and understanding the two new datasets

In [ ]:
df1 = pd.read_csv('sumstats_all_misspe1.csv')
df2 = pd.read_csv('sumstats_all_misspe2.csv')
df1.shape,df2.shape

In [ ]:
df.iloc[:,:11].apply([min,max])

In [ ]:
df1.iloc[:,:11].apply([min,max])

In [ ]:
df2.iloc[:,:11].apply([min,max])

In [ ]:
dfconcat= pd.concat([dfsub_train,df1,df2], keys=["training","misspe1","mispe2"])
dfconcat['dataset'] = dfconcat.index.get_level_values(0)
dfconcat['N1/N2'] = dfconcat['N_1'] /dfconcat['N_2'] 

plt.figure(figsize=(10,10))
plt.subplot(221)
sns.histplot(dfconcat, x='migration_rate', hue='dataset', kde=True)
plt.subplot(222)
sns.histplot(dfconcat, x='T_split', hue='dataset', kde=True)
plt.subplot(223)
sns.histplot(dfconcat, x='N1/N2', hue='dataset')
plt.subplot(224)
sns.histplot(dfconcat, x='Fst_0', hue='dataset', kde=True)

In [ ]:
sns.histplot(dfsub_train.migration_rate, label='training subset')
sns.histplot(df1.migration_rate, label='misspe1')
plt.legend()
plt.figure()
sns.histplot(dfsub_train.migration_rate, label='training subset')
sns.histplot(df2.migration_rate, label='misspe2')
plt.legend()

In [ ]:
print(
    'mean of N1/N2 in train:', np.mean(dfsub_train.N_1/dfsub_train.N_2),
    '; in df1:',np.mean(df1.N_1/df1.N_2),
    '; in df2:',np.mean(df2.N_1/df2.N_2)
)

<div class="alert alert-warning">
    
We have displayed above partial information about the datasets. Can you describe the main differences between these datasets ?    
</div>

ANSWER:

## Using pretrained models to predict the demographic parameters of the two novel datasets

<div class="alert alert-info">
    
Let us apply one of the previously trained (on df_train or dfsub_train) model to these different sets. 
    You can first your favorite model if you want.

We are back to using regression models.

</div>

In [ ]:
n_estimators = 100
my_model = RandomForestRegressor(n_estimators = n_estimators, n_jobs = 4)
target = 'migration_rate'
ss = ss_names

In [ ]:
score, rmse, pred, model =  run_model(my_model, ss, target ,dfsub_train, dfsub_val)

training_score = model.score(df_train[ss],df_train[target])
val_score = model.score(df_val[ss],df_val[target])
df1_score = model.score(df1[ss],df1[target])
df2_score = model.score(df2[ss],df2[target])
print(f'training score: {training_score} \n validation score: {val_score} \n df1 score: {df1_score} \n df2 score: {df2_score}')


Xn, yn = dfsub_val[ss], dfsub_val[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'Validation set {val_score:.3f}', alpha=.2)

Xn, yn = df1[ss], df1[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'misspe1 set {df1_score:.3f}', alpha=.2)

Xn, yn = df2[ss], df2[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'misspe2 set {df2_score:.3f}', alpha=.2)

plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')

plt.axline((0,0),slope=1, color='black', label='y=x')
plt.legend()


In [ ]:
# Same as above but on 3 the different figures
plt.figure(figsize=(15,5))
plt.subplot(131)
Xn, yn = dfsub_val[ss], dfsub_val[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'Validation set {val_score:.3f}')
plt.axline((0,0),slope=1, color='black', label='y=x')

plt.subplot(132)
Xn, yn = df1[ss], df1[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.')
plt.xlabel(f'{target}')
#plt.ylabel(f'Predicted {target}')
plt.title(f'misspe1 set {df1_score:.3f}')
plt.axline((0,0),slope=1, color='black', label='y=x')

plt.subplot(133)
Xn, yn = df2[ss], df2[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.')
plt.xlabel(f'{target}')
#plt.ylabel(f'Predicted {target}')
plt.title(f'misspe2 set {df2_score:.3f}')
plt.axline((0,0),slope=1, color='black', label='y=x')


<div class="alert alert-warning">
    
**What are the different plot representing?**

**Comment on the predictive performances, what do you believe the issue is?**
    
</div>

ANSWER:

<div class="alert alert-warning">

You can go on and test whether other models (rf, linear regrssion, svm, etc) fitted on df_train have the same issue or if some are more robust than others

</div>

In [ ]:
###### Linear regression using all summary statistics
ss = ss_names
target = 'migration_rate'
X = df_train[ss]
y = df_train[target]
reg = LinearRegression().fit(X, y)

In [ ]:
model= reg
training_score = model.score(df_train[ss_names],df_train[target])
val_score = model.score(df_val[ss_names],df_val[target])
df1_score = model.score(df1[ss_names],df1[target])
df2_score = model.score(df2[ss_names],df2[target])
print(f'training score: {training_score} \n validation score: {val_score} \n df1 score: {df1_score} \n df2 score: {df2_score}')


Xn, yn = dfsub_val[ss], dfsub_val[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'Validation set {val_score:.3f}', alpha=.2)

Xn, yn = df1[ss], df1[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'misspe1 set {df1_score:.3f}', alpha=.2)

Xn, yn = df2[ss], df2[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'misspe2 set {df2_score:.3f}', alpha=.2)

plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.legend()


<div class="alert alert-info">
    
**In practice you do not have the labels of real data. So it is hard to check if your target data is outside of your training space (ie the real scenario is not encountered in the prior distribution of the simulations), which is problematic for predictive accuracy as we just saw!**

You can design **robustness experiments** by creating lots of datasets with misspecifications (for which you know the labels) to test the performance of your approach on those, and hope that the robustness conclusion generalizes well to your real data.
    
When using summary statistics a good sanity check is to **investigate whether real summary statistics fall in the range of the summary statistics of the training set/reference panel** (like for Approximate Bayesian Computation), although one expects some ML methods to generalize to out-of-distribution points as well (if they are close-by and if the relationship is smooth enough).
    
*Advanced:* 

Finally it is always a good idea to: 
    
- (i) simulate new data from the inferred model, let's call the model $M_{infer}$
    
- (ii) infer the evolutionnary parameters from these resimulated data yielding the model $M_{reinfer}$
  
- (iii) compare the two models  $M_{infer}$ and  $M_{reinfer}$ 
    
In ABC, you can naturally draw multiple new models  $M^{(i)}_{infer}$ from the posterior distribution. For ML approaches, if you estimated uncertainties or approximated a posterior you can do the same ; if you have a punctual estimator, you can do multiple simulations from it (yielding stochasticity only due to the coalescent stochasticity) or draw it with some small perturbations.
</div>

# 10. Regression with a neural network (MLP Multi Layer Perceptron)
For detailed description of the options and hyperparameters, refer to [the scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPRegressor.html#sklearn.neural_network.MLPRegressor)

### Simple training

<div class="alert alert-info">
We will use a Multi Layer Perceptron (MLP), i.e. a network composed of fully connected layers.

Let's do a first training with one single hidden layer, of a specified size - argument 'hidden_layer_sizes=(50,)'
</div>

In [ ]:
%%time
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(50,))

target = 'migration_rate'
score, rmse, pred, mlp =  run_model(mlp, ss_names, target ,dfsub_train, dfsub_val)
training_score = mlp.score(dfsub_train[ss_names],dfsub_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on validation: {score:.3f} on training: {training_score:.3f}')


<div class="alert alert-warning">
    Clearly something went bad! Do you notice that all the prediction are centered around zero? (y-axis)
</div>

<div class="alert alert-info">
ALthough it is in theory possible for a MLP to learn from unscaled inputs and predict unscaled outputs, it can complicate drastically the optimization procedure (as seen here).
    
Let's train the network on standardized input features and standardized target parameter instead. We'll show how the distribution of the transformed parameter looks like. But scikit learn has a function `TransformedTargetRegressor` that will do automatically the transformation (and inverse transformation after prediction) for you, which is very convenient.

</div>

In [ ]:
# Illustration of the effect of StandardScaler
plt.figure(figsize=(10,5))
y = dfsub_train[[target]]
plt.subplot(121)
sns.histplot(y)
scaler = StandardScaler()
scaler.fit(y)
plt.subplot(122)
sns.histplot(scaler.transform(y), label='Standardized target')
plt.legend()

In [ ]:
%%time
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(50,))
mlp = make_pipeline(StandardScaler(), mlp) # standardize input data (using training info) before doing the MLP regression
mlp = TransformedTargetRegressor(regressor=mlp, transformer=StandardScaler()) # standardize the target variable

target = 'migration_rate'
ss = ss_names
score, rmse, pred, mlp =  run_model(mlp, ss, target ,dfsub_train, dfsub_val)
training_score = mlp.score(dfsub_train[ss],dfsub_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')

plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on training: {training_score:.3f}, on validation: {score:.3f}')



In [ ]:
%%time
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(50,))
mlp = make_pipeline(StandardScaler(), mlp) # standardize input data (using training info) before doing the MLP regression
mlp = TransformedTargetRegressor(regressor=mlp, transformer=StandardScaler()) # standardize the target variable

target = 'migration_rate'
ss = ss_names
score, rmse, pred, mlp =  run_model(mlp, ss, target ,dfsub_train, dfsub_val)
training_score = mlp.score(dfsub_train[ss],dfsub_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')


In [ ]:

plt.figure(figsize=(10,5))
plt.subplot(121)
plt.plot(dfsub_train[target], mlp.predict(dfsub_train[ss]), '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on training: {training_score:.3f}')

plt.subplot(122)
plt.plot(dfsub_val[target], pred, '.')
plt.xlabel(f'{target}')
#plt.ylabel(f'Predicted {target}')
plt.title(f'score on validation: {score:.3f}')


In [ ]:
plt.plot(mlp.regressor_[1].loss_curve_)
plt.xlabel('iteration')
plt.ylabel('Training loss')

<div class="alert alert-warning">

- !NEW PROBLEM! Notice the difference in validation and training scores. The training loss seems to nicely go down though! What do you think is happening?

- The network architecture is defined by the argument passed to MLPRegressor. Here we had MLPRegressor(max_iter=500, hidden_layer_sizes=(50,))

- Given the network architecture (one hidden layer with 50 nodes, one parameter to predict) and the lecture: how many parameters are in this network? (ie weights that are optimized during training?)
</div>

ANSWERS:

<div class="alert alert-info">
We now show below how to access the number of weights directly from the mlp object itself:
</div>

In [ ]:
# you can check the layer sizes by digging into the model 
# first find the regressor block of the pipeline
# then get its coefficients (matrices) and their shape
mlp

In [ ]:
model = mlp.regressor_[1]
model

In [ ]:
 model.coefs_[0].__class__, model.intercepts_[0].__class__

In [ ]:
print('The network has ', len(model.coefs_), 'layers')
print('weights (stored as matrices) - ', 'first (hidden) layer: ' , model.coefs_[0].shape, ', second (output) layer: ', model.coefs_[1].shape)
print('biases (also called intercepts):  - ', 'first (hidden) layer: ' , model.intercepts_[0].shape, ', second (output) layer: ', model.intercepts_[1].shape)

<div class="alert alert-warning">
    
To reduce overfitting one can reduce the number of parameter and/or increase the size of the training datatset (when more data is available)

Let's train a similar network but with one layer of **size 10** and on the **full training set**.

- How many overall optimizable parameters does this new model have?
  
- What's the new performance, is it better or worse?
</div>

In [ ]:
# INSERT CODE BELOW

In [ ]:
%%time
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(10,))
mlp = make_pipeline(StandardScaler(), mlp) # standardize input data (using training info) before doing the MLP regression
mlp = TransformedTargetRegressor(regressor=mlp, transformer=StandardScaler()) # standardize the target variable

target = 'migration_rate'
ss = ss_names
score, rmse, pred, mlp =  run_model(mlp, ss, target ,df_train, df_val)
training_score = mlp.score(df_train[ss],df_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')


In [ ]:

plt.figure(figsize=(10,5))
plt.subplot(121)
plt.plot(dfsub_train[target], mlp.predict(dfsub_train[ss]), '.')
plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.title(f'score on training: {training_score:.3f}')

plt.subplot(122)
plt.plot(df_val[target], pred, '.')
plt.xlabel(f'{target}')
#plt.ylabel(f'Predicted {target}')
plt.title(f'score on validation: {score:.3f}')


### Inspect weights of the one hidden layer network with 10 hidden nodes

In [ ]:
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(10,))
mlp = make_pipeline(StandardScaler(), mlp) # standardize input data (using training info) before doing the MLP regression
mlp = TransformedTargetRegressor(regressor=mlp, transformer=StandardScaler()) # standardize the target variable

target = 'migration_rate'
ss = ss_names
score, rmse, pred, mlp =  run_model(mlp, ss, target ,df_train, df_val)
training_score = mlp.score(df_train[ss],df_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')


In [ ]:
mlp

In [ ]:
mlp.regressor

In [ ]:
# if you click on the small black arrows you can see what are the attributes of this object
mlp.regressor_[1]

In [ ]:
model = mlp.regressor_[1]

nfeat=10

fig, axes = plt.subplots(2, 5, figsize=(15,10))

for node, ax in zip(range(10), axes.ravel()):
    coef = (model.coefs_[0][:,node])
    sorted_idx = coef.argsort()[::-1][:nfeat]
    ax.barh(mlp.feature_names_in_[sorted_idx][::-1], coef[sorted_idx][::-1])
    ax.set_title(f"node {node}")
plt.suptitle(f"{nfeat} larger weight values of node X of first layer")
plt.tight_layout()

plt.figure()
coef = (model.coefs_[1].ravel())
sorted_idx = coef.argsort()[::-1]
plt.barh([f'node {n}' for n in sorted_idx[::-1] ], coef[sorted_idx][::-1])
plt.title('contribution of the hidden nodes to the output layer')

<div class="alert alert-warning">
What are the features used by the top node(s) of the hidden layer contributing to the output layer? </div>

### Feature importance

Explainability methods based on **perturbations** are among the simplest yet useful approaches. They are **model-agnostic** and applicable to almost any framework, in particular black-box models where the internal structure of the model is unknown (unlike the case discussed above, where we could inspect the MLP weights), but where the model can still be queried for predictions. One clear drawback, however, is their computational cost, particularly when the number of features is large.

They rely on the following principle: 
1. perturb the input in a controlled way,
2. use the trained model (which we aim to interpret) to generate predictions for the perturbed input,
3. measure the change in model output or performance between the original and perturbed inputs,
4. the larger this change, the more the model is considered to rely on the perturbed feature(s) for its predictions.

**Permutation importance** is a specific case of this framework, where a feature is randomly shuffled across samples while all other features remain unchanged, thereby breaking its relationship with the target distribution.

In [ ]:
result = permutation_importance(
    mlp,
    dfsub_val[ss],
    dfsub_val[target],
    n_repeats=20,
    random_state=0,
    scoring="r2",
    n_jobs=-1
)

importances = pd.DataFrame({
    "feature": ss,
    "importance_mean": result.importances_mean,
    "importance_std": result.importances_std
}).sort_values("importance_mean", ascending=False)

print(importances.head(20))

<div class="alert alert-warning">
Comment on the results of the permutation importance analysis
</div>

### -skip- Predict multiple target parameters

<div class="alert alert-warning">
    Let's train a network to predict jointly the migration rate and the time of split using e.g. 2 fully connected layers of size 20 and 10.
    It will thus build internal representations that have to be useful to predict these two parameters rather than a single one.
</div>

In [ ]:
%%time
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(20,10))
mlp = make_pipeline(StandardScaler(), mlp) # standardize input data (using training info) before doing the MLP regression
mlp = TransformedTargetRegressor(regressor=mlp, transformer=StandardScaler()) # standardize the target variable

target = ['migration_rate', 'T_split']
ss = ss_names
score, rmse, pred, mlp =  run_model(mlp, ss, target ,df_train, df_val)
training_score = mlp.score(df_train[ss],df_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')


In [ ]:

pred_train = mlp.predict(dfsub_train[ss])

for i,targ in enumerate(target):
    plt.figure(figsize=(10,5))
    plt.subplot(121)
    plt.plot(dfsub_train[targ],pred_train[:,i] , '.',alpha=.2)
    plt.xlabel(f'{targ}')
    plt.ylabel(f'Predicted {targ}')
    plt.title(f'score on training: {training_score:.3f}')

    plt.subplot(122)
    plt.plot(df_val[targ], pred[:,i], '.',alpha=.2)
    plt.xlabel(f'{targ}')
    #plt.ylabel(f'Predicted {targ}')
    plt.title(f'score on validation: {score:.3f}')


### -skip- Impact of training data misspecification

<div class="alert alert-warning">
    
You can test the impact of model mispecification (as in the previous section) on you network, set up some hyper parameter optimization, etc.    
</div>

In [ ]:
# INSERT CODE BELOW

In [ ]:
%%time
mlp = MLPRegressor(max_iter=500, hidden_layer_sizes=(10,))
mlp = make_pipeline(StandardScaler(), mlp) # standardize input data (using training info) before doing the MLP regression
mlp = TransformedTargetRegressor(regressor=mlp, transformer=StandardScaler()) # standardize the target variable

target = 'migration_rate'
ss = ss_names
score, rmse, pred, mlp =  run_model(mlp, ss, target ,df_train, df_val)
training_score = mlp.score(df_train[ss],df_train[target])
print(f'training score: {training_score}')
print(f'validation score: {score}')

model= mlp
training_score = model.score(df_train[ss_names],df_train[target])
val_score = model.score(df_val[ss_names],df_val[target])
df1_score = model.score(df1[ss_names],df1[target])
df2_score = model.score(df2[ss_names],df2[target])
print(f'training score: {training_score} \n validation score: {val_score} \n df1 score: {df1_score} \n df2 score: {df2_score}')


Xn, yn = dfsub_val[ss], dfsub_val[target]
pred = model.predict(Xn)

plt.plot(yn, pred, '.', label=f'Validation set {val_score:.3f}', alpha=.2)

Xn, yn = df1[ss], df1[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'misspe1 set {df1_score:.3f}', alpha=.2)

Xn, yn = df2[ss], df2[target]
pred = model.predict(Xn)
plt.plot(yn, pred, '.', label=f'misspe2 set {df2_score:.3f}', alpha=.2)

plt.xlabel(f'{target}')
plt.ylabel(f'Predicted {target}')
plt.legend()


<div class="alert alert-success">

The problem is still obvious for the dataset misspecification 1 (that has migration rates drastically out of the prior range).

It seems that for the second dataset hower (that has asymetrical population sizes) the performance improved, compared to the random forest previously tested. Of course to draw a solid conclusion one would need to do hyperparameter optimization for both models first, and use a test set that has not been seen during the HP search (contrary to the validation set).

</div>